# 모델 라우팅 품질·비용 비교

같은 분류 문제를 작은 모델과 상위 모델에 실행하고 정확도와 추정 비용을 함께 비교합니다.

## 실행 환경 준비

In [1]:
from pathlib import Path
from dotenv import load_dotenv

here = Path.cwd().resolve()
chapter_root = next((p for p in (here, *here.parents) if p.name == "ch34-cost-rate-limits" and (p / "notebooks").is_dir()), None)
if chapter_root is None:
    raise FileNotFoundError("ch34-cost-rate-limits 실습 폴더 안에서 실행하세요.")
env_path = next((p / ".env" for p in (here, *here.parents) if (p / ".env").exists()), None)
if env_path:
    load_dotenv(env_path)
print("Python 준비 완료", "API 키 파일 확인" if env_path else "API 키 파일 없음")

Python 준비 완료 API 키 파일 확인


## 고정 평가 사례

질문과 정답은 모델 사이에서 바꾸지 않습니다.

In [2]:
cases = [("배송이 사흘째 멈췄어요", "배송"), ("결제를 취소하고 싶어요", "환불"), ("재질이 면인가요", "상품")]
prices = {"openai:gpt-5.4-nano": (0.20, 1.25), "openai:gpt-5.4-mini": (0.75, 4.50)}
print("평가 사례:", len(cases))

평가 사례: 3


## 두 모델 실제 호출

응답은 세 범주 중 하나만 내도록 제한하고 실제 토큰을 기록합니다.

In [3]:
import json
from pathlib import Path
from langchain.chat_models import init_chat_model
results = []
for model in prices:
    current = init_chat_model(model, reasoning_effort="none", max_tokens=20, max_retries=1, seed=34)
    for text, expected in cases:
        response = current.invoke(f"배송, 환불, 상품 중 하나만 답하세요. 문의: {text}")
        answer = response.text.strip()
        usage = response.usage_metadata or {}
        results.append({"model": model, "expected": expected, "answer": answer, "correct": answer == expected,
                        "usage": usage, "input": usage.get("input_tokens", 0), "output": usage.get("output_tokens", 0)})
run_dir = chapter_root / "runs"; run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "routing-comparison.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))
print("실제 호출 수:", len(results))

실제 호출 수: 6


## 정확도와 비용 계산

공식 단가를 백만 토큰당 달러로 적용합니다. 이 작은 실험은 세 사례에서의 결과만 설명합니다.

In [4]:
import json

summary = {}
for model in prices:
    rows = [r for r in results if r["model"] == model]
    inp, out = prices[model]
    cost = sum(r["input"] * inp + r["output"] * out for r in rows) / 1_000_000
    summary[model] = {"accuracy": sum(r["correct"] for r in rows) / len(rows), "cost_usd": cost}
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "openai:gpt-5.4-nano": {
    "accuracy": 0.6666666666666666,
    "cost_usd": 3.325e-05
  },
  "openai:gpt-5.4-mini": {
    "accuracy": 1.0,
    "cost_usd": 0.00012225
  }
}


## 응답 원문 확인

자동 점수만 보지 않고 두 모델이 실제로 반환한 범주를 정답과 나란히 확인합니다.

In [5]:
for row in results:
    print(row["model"], "정답=", row["expected"], "응답=", repr(row["answer"]), "일치=", row["correct"])
assert all(row["answer"] for row in results)

openai:gpt-5.4-nano 정답= 배송 응답= '배송' 일치= True
openai:gpt-5.4-nano 정답= 환불 응답= '환불' 일치= True
openai:gpt-5.4-nano 정답= 상품 응답= '배송' 일치= False
openai:gpt-5.4-mini 정답= 배송 응답= '배송' 일치= True
openai:gpt-5.4-mini 정답= 환불 응답= '환불' 일치= True
openai:gpt-5.4-mini 정답= 상품 응답= '상품' 일치= True


## 라우팅 규칙 적용

짧은 단일 의도 분류는 nano로 보내고, 긴 설명 요청은 mini로 보냅니다.

In [6]:
def route(text):
    complex_request = len(text) > 60 or "비교" in text or "근거" in text
    return "openai:gpt-5.4-mini" if complex_request else "openai:gpt-5.4-nano"

samples = ["배송 상태 알려줘", "환불 규정 두 가지를 비교하고 각 근거를 설명해 주세요."]
print([(text, route(text)) for text in samples])

[('배송 상태 알려줘', 'openai:gpt-5.4-nano'), ('환불 규정 두 가지를 비교하고 각 근거를 설명해 주세요.', 'openai:gpt-5.4-mini')]
